In [14]:
import time
import math
import asyncio
import statistics
import multiprocessing as mp
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from typing import Callable, List, Dict, Any, Tuple
import plotly.graph_objects as go

class BenchmarkEngine:
    """Motor de benchmarking profesional con soporte para warm-up y estadística."""

    @staticmethod
    def run_sync(name: str, func: Callable, args: tuple, iterations: int = 5, warmup: int = 1) -> Dict[str, Any]:
        print(f"[{name}] Ejecutando warm-up ({warmup} iteraciones)...")
        for _ in range(warmup):
            func(*args)

        print(f"[{name}] Ejecutando benchmark ({iterations} iteraciones)...")
        times = []
        for _ in range(iterations):
            t0 = time.perf_counter()
            func(*args)
            times.append(time.perf_counter() - t0)

        return BenchmarkEngine._compile_stats(name, times)

    @staticmethod
    async def run_async(name: str, coro_func: Callable, args: tuple, iterations: int = 5, warmup: int = 1) -> Dict[str, Any]:
        print(f"[{name}] Ejecutando warm-up async ({warmup} iteraciones)...")
        for _ in range(warmup):
            await coro_func(*args)

        print(f"[{name}] Ejecutando benchmark async ({iterations} iteraciones)...")
        times = []
        for _ in range(iterations):
            t0 = time.perf_counter()
            await coro_func(*args)
            times.append(time.perf_counter() - t0)

        return BenchmarkEngine._compile_stats(name, times)

    @staticmethod
    def _compile_stats(name: str, times: List[float]) -> Dict[str, Any]:
        mean_time = statistics.mean(times)
        stdev_time = statistics.stdev(times) if len(times) > 1 else 0.0
        print(f"--> [Resultado] {name}: {mean_time:.3f}s ± {stdev_time:.3f}s\n")
        return {
            "name": name,
            "mean": mean_time,
            "stdev": stdev_time,
            "raw_times": times
        }

1. Análisis de Carga CPU-Bound (Factorización de Primos)
Resultados: El tiempo secuencial fue de 1.77s, mientras que con Threads subió a 2.79s y con Procesos a 7.93s.  

Explicación Técnica:

Impacto del GIL: En Python, el Global Interpreter Lock (GIL) impide que múltiples hilos ejecuten bytecode simultáneamente. Al intentar usar ThreadPoolExecutor, se generó un overhead por el cambio de contexto (context switching) sin ganancia real, empeorando el tiempo.  

Costo de Serialización: El uso de ProcessPoolExecutor fue significativamente más lento (0.2x speedup) debido a la granularidad de la tarea y el costo de comunicación entre procesos (IPC) y la serialización de datos (pickling) para enviar 20,000 números a los workers.

In [15]:
#===========================================
# 1. CARGA CPU-BOUND: Factorización de Primos
# ==========================================
def factorize_number(n: int) -> List[int]:
    """Carga pesada matemática pura. Retorna factores primos."""
    factors = []
    d = 2
    while d * d <= n:
        while (n % d) == 0:
            factors.append(d)
            n //= d
        d += 1
    if n > 1:
        factors.append(n)
    return factors

def workload_cpu_secuencial(numbers: List[int]):
    return [factorize_number(n) for n in numbers]

def workload_cpu_threads(numbers: List[int], workers: int = 4):
    with ThreadPoolExecutor(max_workers=workers) as executor:
        return list(executor.map(factorize_number, numbers))

def workload_cpu_procesos(numbers: List[int], workers: int = 4):
    with ProcessPoolExecutor(max_workers=workers) as executor:
        return list(executor.map(factorize_number, numbers))


2. Análisis de Carga I/O-Bound (Latencia de API)
Resultados: El salto de 4.01s (Secuencial) a 0.30s (Threads), logrando un Speedup de 13.2x.  

Explicación Técnica:

Liberación del GIL: Las operaciones de I/O (como time.sleep o peticiones de red) liberan el GIL mientras esperan la respuesta del sistema operativo. Esto permite que la concurrencia sea efectiva, reduciendo drásticamente la latencia total al solapar los tiempos de espera.

In [16]:
# ==========================================
# 2. CARGA I/O-BOUND: Simulación de Latencia API
# ==========================================
def fetch_mock_api(req_id: int) -> str:
    """Simula una llamada a red bloqueante (ej. requests.get)"""
    time.sleep(0.1) # 100ms de latencia de red
    return f"Response_{req_id}"

def workload_io_secuencial(requests: List[int]):
    return [fetch_mock_api(req) for req in requests]

def workload_io_threads(requests: List[int], workers: int = 16):
    with ThreadPoolExecutor(max_workers=workers) as executor:
        return list(executor.map(fetch_mock_api, requests))



3. Análisis de Carga Mixta (Pipeline ETL)
Resultados: El método Mix_Async_Proc fue el más eficiente con 0.18s, logrando un Speedup de 11.0x frente a los 2.00s secuenciales.  

Explicación Técnica:

Arquitectura Recomendada: Se utilizó un Event Loop de asyncio para manejar la fase de I/O masiva sin bloquear el hilo principal.

Delegación de CPU: La fase pesada de computación se delegó a un ProcessPool mediante run_in_executor. Esta combinación minimiza el bloqueo del loop y aprovecha el paralelismo real de los núcleos para la parte matemática, optimizando el rendimiento global.

In [17]:
# ==========================================
# 3. CARGA MIXTA + ASYNCIO: Pipeline ETL
# ==========================================
async def async_fetch_mock_api(req_id: int) -> str:
    """Simula I/O no bloqueante (ej. aiohttp)"""
    await asyncio.sleep(0.1)
    return f"Data_{req_id}"

async def workload_mixta_async(requests: List[int]):
    # Fase I/O asíncrona masiva
    tasks = [async_fetch_mock_api(req) for req in requests]
    data = await asyncio.gather(*tasks)

    # Fase CPU: Delegamos la CPU pesada a un ProcessPool dentro del Event Loop
    # Esta es la arquitectura profesional recomendada para pipelines mixtos.
    loop = asyncio.get_running_loop()
    with ProcessPoolExecutor(max_workers=4) as pool:
        cpu_tasks = [loop.run_in_executor(pool, factorize_number, 9999999 + i) for i in range(len(data))]
        await asyncio.gather(*cpu_tasks)
    return True

def workload_mixta_secuencial(requests: List[int]):
    for req in requests:
        fetch_mock_api(req)
        factorize_number(9999999 + req)
    return True

In [18]:
# Configuración del volumen de datos
NUMEROS_A_FACTORIZAR = [15485863 + i for i in range(20000)] # 20,000 números grandes
PETICIONES_API = list(range(40)) # 40 llamadas API
PETICIONES_MIXTAS = list(range(20)) # 20 ciclos de ETL

print("=== INICIANDO BENCHMARKS ===")
resultados = {}

# 1. Benchmarks CPU-bound
resultados['cpu_sec'] = BenchmarkEngine.run_sync("CPU_Secuencial", workload_cpu_secuencial, (NUMEROS_A_FACTORIZAR,), 3)
resultados['cpu_thr'] = BenchmarkEngine.run_sync("CPU_Threads", workload_cpu_threads, (NUMEROS_A_FACTORIZAR, 4), 3)
resultados['cpu_proc'] = BenchmarkEngine.run_sync("CPU_Procesos", workload_cpu_procesos, (NUMEROS_A_FACTORIZAR, 4), 3)

# 2. Benchmarks I/O-bound
resultados['io_sec'] = BenchmarkEngine.run_sync("IO_Secuencial", workload_io_secuencial, (PETICIONES_API,), 3)
resultados['io_thr'] = BenchmarkEngine.run_sync("IO_Threads", workload_io_threads, (PETICIONES_API, 16), 3)

# 3. Benchmarks Mixtos (I/O + CPU)
resultados['mix_sec'] = BenchmarkEngine.run_sync("Mix_Secuencial", workload_mixta_secuencial, (PETICIONES_MIXTAS,), 3)
resultados['mix_async'] = await BenchmarkEngine.run_async("Mix_Async_Proc", workload_mixta_async, (PETICIONES_MIXTAS,), 3)

print("=== BENCHMARKS FINALIZADOS ===")

=== INICIANDO BENCHMARKS ===
[CPU_Secuencial] Ejecutando warm-up (1 iteraciones)...
[CPU_Secuencial] Ejecutando benchmark (3 iteraciones)...
--> [Resultado] CPU_Secuencial: 1.719s ± 0.009s

[CPU_Threads] Ejecutando warm-up (1 iteraciones)...
[CPU_Threads] Ejecutando benchmark (3 iteraciones)...
--> [Resultado] CPU_Threads: 2.824s ± 0.766s

[CPU_Procesos] Ejecutando warm-up (1 iteraciones)...
[CPU_Procesos] Ejecutando benchmark (3 iteraciones)...
--> [Resultado] CPU_Procesos: 7.268s ± 0.601s

[IO_Secuencial] Ejecutando warm-up (1 iteraciones)...
[IO_Secuencial] Ejecutando benchmark (3 iteraciones)...
--> [Resultado] IO_Secuencial: 4.009s ± 0.003s

[IO_Threads] Ejecutando warm-up (1 iteraciones)...
[IO_Threads] Ejecutando benchmark (3 iteraciones)...
--> [Resultado] IO_Threads: 0.303s ± 0.001s

[Mix_Secuencial] Ejecutando warm-up (1 iteraciones)...
[Mix_Secuencial] Ejecutando benchmark (3 iteraciones)...
--> [Resultado] Mix_Secuencial: 2.004s ± 0.000s

[Mix_Async_Proc] Ejecutando warm-up

In [12]:
def graficar_resultados_profesionales(res_dict: Dict[str, Dict]):
    """Genera gráficos Plotly interactivos con barras de error y Speedup."""

    # Extraer datos
    escenarios = ['1. CPU-Bound<br>(Factorización Numérica)',
                  '2. I/O-Bound<br>(Latencia Red)',
                  '3. Mixto<br>(ETL: Async I/O + Procesos)']

    # Mapeo de resultados por estrategia (Promedios)
    means = {
        'Secuencial': [res_dict['cpu_sec']['mean'], res_dict['io_sec']['mean'], res_dict['mix_sec']['mean']],
        'Threads': [res_dict['cpu_thr']['mean'], res_dict['io_thr']['mean'], 0],
        'Multiprocessing': [res_dict['cpu_proc']['mean'], 0, 0],
        'Asyncio + Pool': [0, 0, res_dict['mix_async']['mean']]
    }

    # Mapeo de Desviación Estándar (Barras de error)
    stdevs = {
        'Secuencial': [res_dict['cpu_sec']['stdev'], res_dict['io_sec']['stdev'], res_dict['mix_sec']['stdev']],
        'Threads': [res_dict['cpu_thr']['stdev'], res_dict['io_thr']['stdev'], 0],
        'Multiprocessing': [res_dict['cpu_proc']['stdev'], 0, 0],
        'Asyncio + Pool': [0, 0, res_dict['mix_async']['stdev']]
    }

    colores = {'Secuencial': '#34495E', 'Threads': '#3498DB', 'Multiprocessing': '#9B59B6', 'Asyncio + Pool': '#2ECC71'}

    fig = go.Figure()

    for estrategia, tiempos in means.items():
        errores = stdevs[estrategia]
        text_labels = []

        # Calcular Speedup para las anotaciones
        for i, t in enumerate(tiempos):
            if t == 0:
                text_labels.append("")
                continue

            base_time = means['Secuencial'][i]
            speedup = base_time / t

            if estrategia == 'Secuencial':
                text_labels.append(f"<b>{t:.2f}s</b>")
            else:
                text_labels.append(f"<b>{t:.2f}s</b><br>(<b>{speedup:.1f}x</b>)")

        fig.add_trace(go.Bar(
            name=estrategia,
            x=escenarios,
            y=tiempos,
            text=text_labels,
            textposition='outside',
            marker_color=colores[estrategia],
            error_y=dict(type='data', array=errores, visible=True, color='rgba(0,0,0,0.4)', thickness=1.5),
            hovertemplate="Estrategia: %{data.name}<br>Escenario: %{x}<br>Tiempo: %{y:.3f}s ± %{error_y.array:.3f}s<extra></extra>"
        ))

    fig.update_layout(
        title=dict(
            text='<b>Análisis de Rendimiento Concurrente en Arquitectura CPython</b><br><sup>Evaluación estadística (n=3) de impacto del GIL, IPC y Event Loops en tareas de Ciencia de Datos</sup>',
            font=dict(size=18, family="Arial")
        ),
        yaxis=dict(title='<b>Tiempo de Ejecución Medio (Segundos)</b>', gridcolor='#e5e5e5', zerolinecolor='#e5e5e5'),
        xaxis=dict(title='<b>Tipología del Cuello de Botella</b>', tickfont=dict(size=13)),
        barmode='group',
        plot_bgcolor='white',
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1, bgcolor='rgba(255,255,255,0.8)'),
        margin=dict(t=120, b=60, l=60, r=30),
        height=650
    )

    # Extender el eje Y un poco para que las barras de error y el texto no se corten
    max_val = max([max(lst) for lst in means.values()])
    fig.update_yaxes(range=[0, max_val * 1.25])

    fig.show()

# Ejecutar el ploteo
graficar_resultados_profesionales(resultados)

 Conclusion

 Ley de rendimientos decrecientes: Para tareas CPU-bound con datos pequeños/medianos, el paralelismo en Python puede ser contraproducente por el costo de gestión de procesos.

Eficacia de Asyncio: La programación asíncrona es superior para manejar altas densidades de tareas I/O-bound comparado con el modelo de hilos tradicional.

Híbridos: La mejor arquitectura para Ciencia de Datos mixta es mantener la orquestación en un Event Loop y las tareas pesadas en procesos independientes.